# 01. GitHub Repository Ingestion

This notebook demonstrates the repository ingestion stage of the RAG
pipeline.

Given a public GitHub repository URL, the `GitHubRepositoryLoader`
extracts the repository files and represents them as LangChain
`Document` objects.

The notebook explores:

- The number and types of extracted files.
- The structure of a `Document`.
- The document content.
- The metadata associated with each document.

In [15]:
from collections import Counter
import pandas as pd
from github_rag.ingestion.repository import GitHubRepositoryLoader

Set the GitHub repository to explore.

In [16]:
# I use my own repository about Explainable AI for urban typology classification
REPOSITORY_URL = "https://github.com/davidfernxndez/urban-typology-xai"

## 1. Load a GitHub Repository

In [17]:
# Set the Loader Object
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Documents loaded: {len(documents)}")

Documents loaded: 28


In [18]:
file_types = Counter(
    document.metadata["file_type"]
    for document in documents
)

file_types_df = pd.DataFrame(
    file_types.items(),
    columns=["File type", "Documents"],
).sort_values(
    "Documents",
    ascending=False,
)

file_types_df

,File type,Documents
4,.py,13
5,.ipynb,9
2,.txt,2
0,.toml,2
1,.md,1
3,.yml,1


## 2. Exploring LangChain Documents

The ingestion stage returns a list of LangChain `Document` objects.

Each `Document` contains two main components:
- `page_content`: the textual content of the file.
- `metadata`: information about the document's origin and type.

In [19]:
# Select a document to explore
document = documents[0]
print(type(document))

<class 'langchain_core.documents.base.Document'>


In [20]:
print("Page content:")
print(document.page_content[:1000])

Page content:
[build-system]
requires = ["setuptools>=61.0.0"]
build-backend = "setuptools.build_meta"

[project]
name = "urban-typology-xai"
version = "0.1.0"
description = "Explainable machine learning for the classification of residential enclosure patterns"
readme = "README.md"
requires-python = ">=3.11"

[tool.setuptools]
packages = ["src"]


In [22]:
print("\nMetadata:")
document.metadata


Metadata:


{'repository': 'davidfernxndez/urban-typology-xai',
 'file_path': 'pyproject.toml',
 'file_type': '.toml',
 'source': 'https://github.com/davidfernxndez/urban-typology-xai/blob/main/pyproject.toml',
 'branch': 'main'}

Inspect more documents:

In [23]:
for i, document in enumerate(documents[1:5]):
    print(f"--- Document {i} ---")
    print(f"File: {document.metadata['file_path']}")
    print(f"Type: {document.metadata['file_type']}")
    print(f"Source: {document.metadata['source']}")
    print(f"Content preview:\n{document.page_content[:300]}")
    print()

--- Document 0 ---
File: README.md
Type: .md
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/README.md
Content preview:
# Explainable Artificial Intelligence for the Study of Urban Fragmentation

### A Comparative Analysis of the Performance and Interpretability of Supervised Learning Models for the Classification of Residential Enclosure Patterns

<p align="left">
  <img src="https://img.shields.io/badge/Language-Py

--- Document 1 ---
File: requirements-full.txt
Type: .txt
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/requirements-full.txt
Content preview:
# ==============================================================================
# COMPLETE DEVELOPMENT CONFIGURATION (LOCAL)
#
# This file contains all the project dependencies, covering both the
# experimentation phase and the application of Streamlit.
# ===========================================

--- Document 2 ---
File: requirements.txt
Type: .txt
Source: https://github.com/david

As we can see, the ingestion stage stores the following metadata:
- **repository**: repository name.
- **file_path**: path of the file within the repository.
- **file_type**: file extension.
- **source**: direct GitHub URL of the file.
- **branch**: checked-out repository branch.

This metadata will later be preserved throughout the RAG pipeline and used to identify the source of retrieved chunks.